# Previsao de crescimento de gramineas - faixa de dominio

Notebook de inferencia. Roda as celulas **na ordem**.

Correcoes desta versao:
1. **Quantis ordenados** (`np.sort` linha a linha). Modelos de quantil sao independentes
   e podem se cruzar; sem isso o q50 sai acima do q90.
2. **Upload sempre pedido** - a versao antiga pulava o upload se ja existisse um
   `modelo_gramas.pkl` na sessao e carregava o modelo velho em silencio.
3. **Guardas de dominio** - avisa quando os inputs caem fora do espaco de treino
   (ex.: `altura_inicial_cm` abaixo do residuo de rocada) ou quando o intervalo
   sai degenerado.
4. **1 chamada de clima** - as celulas de sensibilidade reaproveitam a janela ja
   baixada em vez de bater na Open-Meteo 8 vezes.

## 1) Modelo

In [ ]:
# Sempre pede o arquivo: assim voce nunca fica com um pkl velho por engano.
import os, joblib, sklearn
from google.colab import files

print("Selecione o modelo_gramas.pkl gerado pelo treinar_modelo.py:")
up = files.upload()
ARQ = "modelo_gramas.pkl" if "modelo_gramas.pkl" in up else list(up)[0]

pk = joblib.load(ARQ)

print()
print("arquivo          :", ARQ)
print("sklearn no Colab :", sklearn.__version__)
print("modelo treinado  :", pk["treinado_em"], "   <- confira que e o modelo novo")
print("linhas de treino :", format(pk["n_linhas"], ",").replace(",", "."))
print("dataset de origem:", pk["fonte"])
print("alvo             :", pk["alvo"])
print("quantis          :", pk["quantis"])
print("R2 locais novos  :", round(pk["metricas"]["r2_locais_novos"], 3),
      "| MAE", round(pk["metricas"]["mae_locais_novos"], 2), "cm")
print("cobertura q10-q90:", str(round(pk["metricas"]["cobertura_q10_q90"] * 100)) + "%")
print()
print(pk["aviso"])

## 2) Preenchedor: clima real + balanco de agua + features

Contem o `prever()` **corrigido** (com `np.sort`) e o `checar()` de dominio.
Se voce editar esta celula, precisa **executa-la de novo** antes das seguintes.

In [ ]:
import json, urllib.parse, urllib.request
from datetime import date, timedelta
import numpy as np, pandas as pd

ARCHIVE  = "https://archive-api.open-meteo.com/v1/archive"
FORECAST = "https://api.open-meteo.com/v1/forecast"
DIARIAS  = ("temperature_2m_mean,temperature_2m_min,temperature_2m_max,"
            "relative_humidity_2m_mean,precipitation_sum,"
            "shortwave_radiation_sum,et0_fao_evapotranspiration")
AQUECIMENTO, LAG_ARCHIVE = 120, 6

ESPECIES = {
    "braquiaria": dict(t_base=15., t_ot2=35., geada=2.0,
                       flor_meses=(2, 3, 4), flor_hmin=30.),
    "esmeralda":  dict(t_base=12., t_ot2=32., geada=-2.0,
                       flor_meses=(), flor_hmin=99.),
    "batatais":   dict(t_base=13., t_ot2=33., geada=0.0,
                       flor_meses=(11, 12, 1, 2, 3), flor_hmin=10.),
}

def _get(u):
    with urllib.request.urlopen(u, timeout=120) as r:
        return json.load(r)

def clima_diario(lat, lon, d0, d1):
    hoje = date.today()
    corte = hoje - timedelta(days=LAG_ARCHIVE)
    partes, fontes = [], []
    a1 = min(d1, corte)
    if d0 <= a1:
        js = _get(ARCHIVE + "?" + urllib.parse.urlencode({
            "latitude": lat, "longitude": lon,
            "start_date": d0.isoformat(), "end_date": a1.isoformat(),
            "daily": DIARIAS, "timezone": "America/Sao_Paulo"}))
        assert "daily" in js, js
        partes.append(pd.DataFrame(js["daily"]))
        fontes.append("archive")
    if d1 > a1:
        past = max(0, (hoje - (a1 + timedelta(days=1))).days + 1)
        fut = max(0, (d1 - hoje).days)
        assert fut <= 16, "janela vai longe demais no futuro"
        js = _get(FORECAST + "?" + urllib.parse.urlencode({
            "latitude": lat, "longitude": lon, "daily": DIARIAS,
            "past_days": min(past, 92), "forecast_days": max(fut, 1),
            "timezone": "America/Sao_Paulo"}))
        assert "daily" in js, js
        f = pd.DataFrame(js["daily"])
        f = f[(f.time >= (a1 + timedelta(days=1)).isoformat()) &
              (f.time <= d1.isoformat())]
        partes.append(f)
        fontes.append("forecast")
    d = pd.concat(partes, ignore_index=True).rename(columns={
        "time": "data", "temperature_2m_mean": "tmed",
        "temperature_2m_min": "tmin", "temperature_2m_max": "tmax",
        "relative_humidity_2m_mean": "umid", "precipitation_sum": "chuva",
        "shortwave_radiation_sum": "rad",
        "et0_fao_evapotranspiration": "et0"})
    d["data"] = pd.to_datetime(d.data)
    d = d.drop_duplicates("data").sort_values("data").reset_index(drop=True)
    for c in ("tmed", "tmin", "tmax", "umid", "chuva", "rad", "et0"):
        d[c] = pd.to_numeric(d[c], errors="coerce")
    cols = ["tmed", "tmin", "tmax", "umid", "rad", "et0"]
    d[cols] = d[cols].ffill().bfill()
    d["chuva"] = d.chuva.fillna(0.)
    return d, "+".join(fontes)

def balanco_solo(clima, cap, h0):
    sw = cap * 0.6
    frac = np.empty(len(clima))
    ench = np.zeros(len(clima), bool)
    consec = 0
    kc = 0.4 + 0.6 * min(max(h0, 0.) / 40., 1.)
    for i, r in enumerate(clima.itertuples()):
        sw = min(sw + r.chuva, cap)
        ks = min(max(sw / (0.55 * cap), 0.), 1.)
        sw = min(max(sw - r.et0 * kc * ks, 0.), cap)
        f = sw / cap
        consec = consec + 1 if (f > 0.97 and r.chuva > 8.) else 0
        frac[i] = f
        ench[i] = consec >= 3
    return frac, ench

def montar(esp, h0, lat, lon, d0, d1, dias_roc, fert=0.35, cap=60.):
    """Janela [d0, d1) - o dia d1 NAO entra. [12/08, 16/08) = 4 dias."""
    e = ESPECIES[esp]
    clima, fonte = clima_diario(lat, lon, d0 - timedelta(days=AQUECIMENTO), d1)
    frac, ench = balanco_solo(clima, cap, h0)
    jan = (clima.data.dt.date >= d0) & (clima.data.dt.date < d1)
    c = clima[jan]
    fr = frac[jan.to_numpy()]
    en = ench[jan.to_numpy()]
    gdd = float(np.clip(np.minimum(c.tmed, e["t_ot2"]) - e["t_base"], 0, None).sum())
    flor = int(((c.data.dt.month.isin(e["flor_meses"])) & (h0 > e["flor_hmin"])).sum())
    linha = dict(
        especie=esp, dias_periodo=int(jan.sum()), altura_inicial_cm=h0,
        dias_desde_rocada_inicio=dias_roc,
        temperatura_media_c=round(float(c.tmed.mean()), 1),
        temperatura_min_c=round(float(c.tmin.min()), 1),
        temperatura_max_c=round(float(c.tmax.max()), 1),
        graus_dia_acumulados=round(gdd, 1),
        umidade_media_pct=round(float(c.umid.mean()), 1),
        precipitacao_total_mm=round(float(c.chuva.sum()), 1),
        dias_com_chuva=int((c.chuva > 1.).sum()),
        et0_medio_mm_dia=round(float(c.et0.mean()), 2),
        radiacao_media_mj_m2=round(float(c.rad.mean()), 1),
        agua_solo_media_pct=round(float(fr.mean() * 100), 1),
        capacidade_agua_solo_mm=cap, fertilidade_solo=fert,
        latitude=round(lat, 4),
        geadas_no_periodo=int((c.tmin <= e["geada"]).sum()),
        dias_encharcado=int(en.sum()), dias_floracao=flor)
    return linha, fonte, c

def prever(linhas):
    """CORRIGIDO: os 3 modelos de quantil sao independentes e podem se cruzar.
    O np.sort linha a linha garante q10 <= q50 <= q90 em qualquer cenario."""
    df = pd.DataFrame(linhas)
    faltando = [c for c in pk["features"] if c not in df.columns]
    if faltando:
        raise ValueError(f"faltam features: {faltando}")
    X = df[pk["features"]].copy()
    X["especie"] = pd.Categorical(X["especie"], categories=pk["categorias"])
    qs = sorted(pk["quantis"])
    Q = np.sort(np.column_stack([pk["modelos"][q].predict(X) for q in qs]), axis=1)
    for i, q in enumerate(qs):
        df["q" + str(int(q * 100))] = np.round(Q[:, i], 2)
    return df

def checar(linha, q10, q50, q90):
    """Guardas de dominio: avisa quando o cenario cai fora do espaco de treino."""
    av = []
    h0, roc = linha["altura_inicial_cm"], linha["dias_desde_rocada_inicio"]
    if h0 < 4:
        av.append(f"altura_inicial_cm={h0}: abaixo do residuo de rocada do treino "
                  "(5-12 cm). O modelo esta EXTRAPOLANDO - resultado nao confiavel.")
    if h0 > 4 and roc == 0 and h0 > 20:
        av.append(f"h0={h0} cm com dias_desde_rocada=0: rocada nao deixa a planta "
                  "tao alta. Confira se os dois inputs batem entre si.")
    if roc > 60 and h0 < 8:
        av.append(f"roc={roc} dias com h0={h0} cm: combinacao rara no treino.")
    li, ls = q50 - q10, q90 - q50
    if li > 0 and ls < 0.3 * li:
        av.append("intervalo degenerado (q50 colado no q90): regiao rala do dataset.")
    if li > 0 and li < 0.3 * ls:
        av.append("intervalo degenerado (q50 colado no q10): regiao rala do dataset.")
    return av

def mostrar(linha, r, fonte=None, rotulo=""):
    q10, q50, q90 = float(r.q10), float(r.q50), float(r.q90)
    h0 = linha["altura_inicial_cm"]
    assert q10 <= q50 <= q90, "quantis cruzados - o np.sort nao rodou!"
    print(f"{linha['especie']} | {int(linha['dias_periodo'])} dias {rotulo}")
    print()
    print("  crescimento provavel (q50) : %+.2f cm" % q50)
    print("  intervalo 80%% (q10-q90)    : %+.2f a %+.2f cm" % (q10, q90))
    print("  altura final estimada      : %.1f cm  (%.1f a %.1f)"
          % (h0 + q50, h0 + q10, h0 + q90))
    for a in checar(linha, q10, q50, q90):
        print("\n  [AVISO] " + a)
    if fonte and "forecast" in fonte:
        print()
        print("  AVISO: parte do clima veio de PREVISAO do tempo;")
        print("  a incerteza real e maior que o intervalo acima.")
    return q10, q50, q90

print("preenchedor pronto | prever() com np.sort:",
      "sort" in prever.__code__.co_names)

## 3) O seu caso

`D1` e **exclusivo**: `[12/08, 16/08)` = 4 dias (12, 13, 14, 15).
`DIAS_ROC = 0` significa cortada no primeiro dia da janela.

In [ ]:
LAT, LON = -21.277917, -43.177872
D0, D1   = date(2026, 8, 12), date(2026, 8, 16)
H0       = 10.0     # altura logo apos o corte (cm)
DIAS_ROC = 0.0      # dias desde a rocada, no INICIO da janela
REGIME   = "pasto"  # "faixa" (rodovia) ou "pasto": muda a profundidade de raiz
FERT     = 0.35     # premissa. O mapa deu 0,404 neste ponto (N 1,71 g/kg)
# O balde do SoilGrids neste ponto, nas duas profundidades de raiz do solo.py:
# 500 mm em faixa de dominio (solo decapitado), 800 em pasto.
CAP      = {"faixa": 59.1, "pasto": 94.6}[REGIME]

linha, fonte, clima_jan = montar("braquiaria", H0, LAT, LON, D0, D1, DIAS_ROC,
                                 fert=FERT, cap=CAP)

print("fonte do clima:", fonte)
print()
print("clima dia a dia da janela:")
print(clima_jan[["data", "tmed", "tmin", "tmax", "chuva", "umid", "et0"]]
      .to_string(index=False))
print()
print("features montadas:")
for k, v in linha.items():
    print("  " + k.ljust(32), v)

# sanidade dos inputs, antes de prever
if int(linha["dias_periodo"]) != (D1 - D0).days:
    print("\n[AVISO] dias_periodo =", linha["dias_periodo"],
          "mas a janela pedida tem", (D1 - D0).days, "dias (falta clima?)")
if linha["altura_inicial_cm"] != H0:
    print("\n[AVISO] altura_inicial_cm nao bate com H0")

## 4) Previsao

In [ ]:
df = prever([linha])
r  = df.iloc[0]

q10, q50, q90 = mostrar(linha, r, fonte, rotulo="| recem-rocada")

## 5) Sensibilidade: fase do ciclo (dias desde a rocada)

Mesma planta, mesmo clima - so muda a fase da curva de rebrota.
Reaproveita a janela ja baixada: **nenhuma chamada nova a Open-Meteo**.

In [ ]:
dias_lista = [0, 2, 5, 10, 20, 40, 70, 120]
linhas = [dict(linha, dias_desde_rocada_inicio=float(d)) for d in dias_lista]

s = prever(linhas)
s["dias_desde_rocada"] = dias_lista
s["altura_final_q50"]  = (H0 + s.q50).round(1)
print(s[["dias_desde_rocada", "q10", "q50", "q90", "altura_final_q50"]]
      .to_string(index=False))
print()
print("A MESMA planta, no MESMO clima, cresce em ritmos diferentes conforme")
print("a fase do ciclo. Logo apos o corte o alongamento vem de reservas;")
print("na fase linear ele e maximo; perto do teto do sitio ele satura.")

## 6) Sensibilidade: fertilidade do micrositio

A faixa de dominio nao e homogenea: valeta umida e pe de talude acumulam
materia organica e crescem muito mais que o meio do plato. `fertilidade_solo`
e a premissa menos observavel do pipeline - vale ver o quanto ela move.

In [ ]:
ferts = [0.20, 0.35, 0.50, 0.70, 0.90]
linhas_f = [dict(linha, fertilidade_solo=f) for f in ferts]

sf = prever(linhas_f)
sf["fertilidade"] = ferts
print(sf[["fertilidade", "q10", "q50", "q90"]].to_string(index=False))

## 7) Confronto com a medicao de campo

In [ ]:
OBSERVADO = 7.0   # cm medidos entre as duas visitas

print("observado em campo : %+.2f cm  (%.2f cm/dia)"
      % (OBSERVADO, OBSERVADO / linha["dias_periodo"]))
print("modelo (q50)       : %+.2f cm" % q50)
print("intervalo 80%%      : %+.2f a %+.2f cm" % (q10, q90))
print()
dentro = q10 <= OBSERVADO <= q90
print("observacao dentro do intervalo?", "SIM" if dentro else "NAO")
if not dentro:
    print("  excede o q90 em %+.2f cm" % (OBSERVADO - q90))
    print("""
JA TESTADO E DESCARTADO -- nao recomece por aqui

 1. FERTILIDADE. Descartada com numero. Varrer de 0,05 a 0,978 (o maximo que o
    modelo viu) move o q50 desta janela de +1,64 a +3,81 cm: a escala INTEIRA
    da feature vale 2,2 cm, e a diferenca a explicar e 4,5. A fisica do gerador
    diz o mesmo por outro caminho: f_N vai de 0,55 a 1,0, um fator de 1,8,
    quando seria preciso 4,2.
 2. CAPACIDADE DE AGUA. Descartada, e com o sinal ao contrario do esperado:
    balde mais fundo PIORA a previsao nesta janela (35 mm -> +2,29 cm;
    120 mm -> +1,29), porque ks e fracao da capacidade e a mesma chuva enche
    menos um balde maior.
 3. AMOSTRAGEM. Era a hipotese da v3.2, que adensou a celula rasa e retreinou.
    A previsao do caso foi de +3,89 para +2,82: o modelo ficou MELHOR e a
    diferenca continuou.
 4. O TETO `A`. A v3.1 subiu de 1,60 para 1,90 cm/dia para alcancar este ponto,
    e a v3.3 DESFEZ. Com f_N = 1, zero estresse hidrico e sitio mediano, a
    fisica desta janela da +3,30 cm; para dar +7 seria preciso A = 4,03 cm/dia
    de DOSSEL -- mais que o dobro da TAlF maxima do marandu, que foi de onde o
    1,90 saiu. Um teto de dossel calibrado por medicao de folha infla toda
    previsao do painel, e e o painel que agenda rocada.

O QUE SOBRA: A GRANDEZA MEDIDA

 1,75 cm/dia e exatamente o teto da TAlF do marandu (12,4-17,5 mm/dia). A folha
 mais alta de uma touceira recem-cortada bater nesse teto e o resultado
 ESPERADO -- e nao e a mesma grandeza que o alvo do modelo, que e altura de
 DOSSEL. O resto cabe no micrositio: com o solo deste ponto seriam +5,8 sigma,
 e em pasto existe mecanismo para isso (mancha de urina ou de esterco, sombra
 de arvore). A dispersao entre 5 touceiras mede esse sigma direto.

PROTOCOLO -- vale mais que dez outliers
 - 5 ou mais pontos, altura MEDIA do dossel, nao a folha mais alta
 - zero da trena no SOLO, manta de folha seca afastada, e o zero DENTRO da foto
 - mesmo instrumento nas duas visitas, mesma hora do dia
 - anotar se choveu nas 24 h antes de cada leitura
 - em pasto, horizonte acima de ~30 dias exige GAIOLA DE EXCLUSAO: piquete nao
   passa 60 dias intocado, e janela com pastejo dentro nao mede crescimento
 - procurar bosta perto da touceira, e medir touceiras a 2-5 m de distancia

E REGISTRE a medicao em `validacao_campo.json`, com o protocolo declarado.
`python validar_campo.py` confronta cada observacao com o modelo e diz quais
sao comparaveis com o alvo dele. Uma constante nesta celula nao guarda o
protocolo -- foi assim que duas versoes do gerador foram calibradas contra uma
leitura de grandeza desconhecida.""")

## 8) Autoteste

Roda quando algo parecer estranho. Mostra a saida crua do modelo, antes de
qualquer tratamento.

In [ ]:
X = pd.DataFrame([linha])[pk["features"]].copy()
X["especie"] = pd.Categorical(X["especie"], categories=pk["categorias"])
qs = sorted(pk["quantis"])
bruto = np.column_stack([pk["modelos"][q].predict(X) for q in qs])[0]

print("modelo carregado :", pk["treinado_em"], "|",
      format(pk["n_linhas"], ",").replace(",", "."), "linhas")
print("quantis no pkl   :", pk["quantis"])
print("saida BRUTA      :", np.round(bruto, 2))
print("saida ORDENADA   :", np.round(np.sort(bruto), 2))
print("o que prever() da:", prever([linha])[["q10", "q50", "q90"]].values[0])
print()
print("prever() tem np.sort?", "sort" in prever.__code__.co_names)
print("houve cruzamento neste cenario?",
      "SIM - o sort corrigiu" if not np.all(np.diff(bruto) >= 0) else "nao")
print()
print("Se a BRUTA vier cruzada e a de prever() vier ordenada, esta tudo certo.")
print("Se prever() vier cruzada, a celula 2 nao foi executada apos a edicao.")